In [1]:
import pandas as pd
import numpy as np
from collections import Counter

!pip install pandasql -q
from pandasql import sqldf
q = lambda sql: sqldf(sql, globals())

  Preparing metadata (setup.py) ... done


In [3]:
df = pd.read_excel('/content/Case - Tarmac Technologies.xlsx', sheet_name='Data')

# DATA INTEGRITY FINDINGS (SQL)

In [9]:
#1. Null rates
print("\n1. Null Rates:")
print(q("""
    SELECT 'STA / ATA' AS field,
        SUM(CASE WHEN sta IS NULL THEN 1 ELSE 0 END) AS null_count,
        ROUND(SUM(CASE WHEN sta IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS null_rate,
        'Departure-only flights' AS reason
    FROM df
    UNION ALL
    SELECT 'ADC',
        SUM(CASE WHEN adc IS NULL THEN 1 ELSE 0 END),
        ROUND(SUM(CASE WHEN adc IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1),
        'Recording failure'
    FROM df
    UNION ALL
    SELECT 'planning_start',
        SUM(CASE WHEN planning_start IS NULL THEN 1 ELSE 0 END),
        ROUND(SUM(CASE WHEN planning_start IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1),
        'Checkpoint tasks (by design)'
    FROM df
""").to_string(index=False))


1. Null Rates:
         field  null_count  null_rate                       reason
     STA / ATA         849       15.3       Departure-only flights
           ADC         101        1.8            Recording failure
planning_start        3359       60.4 Checkpoint tasks (by design)


In [8]:
#2 is_punctual bug
print("\n2. is_punctual by task type:")
print(q("""
    SELECT
        CASE WHEN planning_start IS NULL THEN 'Checkpoint' ELSE 'Scheduled' END AS task_type,
        CASE WHEN is_punctual = 1 THEN 'True' ELSE 'False' END AS is_punctual,
        COUNT(*) AS rows
    FROM df
    GROUP BY task_type, is_punctual
    ORDER BY task_type
""").to_string(index=False))

print("\nCheckpoint marked True but actually late:")
print(q("""
    SELECT COUNT(*) AS late_but_marked_true
    FROM (SELECT DISTINCT turnaround_id, task_name, planning_end, actual_end, is_punctual, planning_start FROM df)
    WHERE planning_start IS NULL AND is_punctual = 1 AND actual_end > planning_end
""").to_string(index=False))


2. is_punctual by task type:
 task_type is_punctual  rows
Checkpoint        True  3359
 Scheduled       False  2205

Checkpoint marked True but actually late:
 late_but_marked_true
                  149


In [10]:
# 3 task_is_applicable
print("\n3 task_is_applicable:")
print(q("""
    SELECT CASE WHEN task_is_applicable = 1 THEN 'True' ELSE 'False' END AS value,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM df), 1) AS pct
    FROM df GROUP BY task_is_applicable
""").to_string(index=False))


3 task_is_applicable:
value  count   pct
 True   5564 100.0


In [11]:
# 4 Free text standardization ---
print("\n4 Risk Identified values:")
print(q("""
    SELECT text_value, COUNT(*) AS count,
        CASE WHEN LOWER(text_value) IN ('nil', 'n', 'ras', 'no') THEN 'No risk'
             ELSE 'Actual risk' END AS meaning
    FROM df
    WHERE custom_label = 'Risk Identified' AND text_value IS NOT NULL
    GROUP BY text_value ORDER BY count DESC
""").to_string(index=False))


4 Risk Identified values:
     text_value  count     meaning
            nil      4     No risk
            NIL      3     No risk
           +2+8      2 Actual risk
            ras      1     No risk
avi fwd +15/+25      1 Actual risk
             No      1     No risk
              N      1     No risk
           FRET      1 Actual risk
          3avih      1 Actual risk
